# 04c — Model Context Protocol (MCP)
**Domain:** IT / DevOps Tools &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** MCP server, typed tools, FastMCP, Inspector

---
### What is MCP?
MCP is an open standard that separates **tool definition** (server) from **tool use** (client).  
Any MCP-compatible LLM client (Claude Desktop, LM Studio, custom apps) can call your tools.

```
LLM Client  ←─ MCP protocol ─→  MCP Server  ←→  External systems
```


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
try:
    import fastmcp
    print("✅ FastMCP available")
except ImportError:
    print("❌ pip install fastmcp mcp")


---
## 🔵 Example — Ex 04c-1: Minimal MCP Server

In [ ]:
# Write mcp_server_minimal.py — run it in a terminal to test
MINIMAL_CODE = (
    'from fastmcp import FastMCP\n'
    'import random\n\n'
    'mcp = FastMCP("DevOps Assistant")\n\n'
    '@mcp.tool()\n'
    'def get_server_status(server_name: str) -> dict:\n'
    '    """Get current status of a server."""\n'
    '    return {"server": server_name,\n'
    '            "status": random.choice(["healthy", "degraded", "down"]),\n'
    '            "cpu_pct": random.randint(10, 95),\n'
    '            "mem_pct": random.randint(20, 90)}\n\n'
    '@mcp.tool()\n'
    'def list_alerts(severity: str = "all") -> list:\n'
    '    """List current operational alerts. Filter: high|medium|low|all."""\n'
    '    alerts = [\n'
    '        {"id": "ALT-001", "severity": "high",   "msg": "DB latency spike"},\n'
    '        {"id": "ALT-002", "severity": "medium", "msg": "Disk 75% full"},\n'
    '        {"id": "ALT-003", "severity": "low",    "msg": "Cert expires in 30d"},\n'
    '    ]\n'
    '    return alerts if severity == "all" else [a for a in alerts if a["severity"] == severity]\n\n'
    'if __name__ == "__main__":\n'
    '    mcp.run()\n'
)

with open('mcp_server_minimal.py', 'w') as f:
    f.write(MINIMAL_CODE)

print('✅ mcp_server_minimal.py written')
print('Run in terminal:  python mcp_server_minimal.py')
print('Inspect:          npx @modelcontextprotocol/inspector python mcp_server_minimal.py')


---
## 🟡 Exercise — Ex 04c-2: DevOps MCP Server with 4 Tools

In [ ]:
show_exercise(
    "04c-2", "DevOps MCP server with 4 tools",
    """Build devops_mcp_server.py (FastMCP) with tools:
  get_server_status(server_name)
  run_health_check(service, timeout_seconds)
  list_alerts(severity)
  create_incident_ticket(title, severity, description) → returns ticket_id
All tools must have type-annotated params and docstrings (FastMCP auto-generates schemas).""",
    hints=[
        "@mcp.tool() decorator — FastMCP builds the JSON schema from type hints + docstring",
        "Use an in-memory dict to store tickets; increment a counter for ticket_id",
        "run_health_check: simulate response_time_ms and status (up/down)",
    ],
    checks=[
        "devops_mcp_server.py has >= 4 @mcp.tool() functions",
        "All tools have docstrings and type-annotated parameters",
        "create_incident_ticket returns a dict with 'ticket_id'",
    ],
)


In [ ]:
# devops_mcp_server.py scaffold — fill in the TODOs!
DEVOPS_LINES = [
    'from fastmcp import FastMCP',
    'from typing import Optional',
    'import random, datetime',
    '',
    'mcp = FastMCP("DevOps MCP Server")',
    '_tickets = {}',
    '_counter = [1000]',
    '',
    '',
    '@mcp.tool()',
    'def get_server_status(server_name: str) -> dict:',
    '    """Get CPU, memory and overall status of a named server."""',
    '    # TODO: return dict with server, status, cpu_pct, mem_pct',
    '    pass',
    '',
    '',
    '@mcp.tool()',
    'def run_health_check(service: str, timeout_seconds: int = 5) -> dict:',
    '    """Run a health check. Returns response_time_ms and status."""',
    '    # TODO: simulate response time and up/down status',
    '    pass',
    '',
    '',
    '@mcp.tool()',
    'def list_alerts(severity: Optional[str] = None) -> list:',
    '    """List operational alerts. Optional filter: high|medium|low."""',
    '    # TODO: return a realistic list of alert dicts',
    '    pass',
    '',
    '',
    '@mcp.tool()',
    'def create_incident_ticket(title: str, severity: str, description: str) -> dict:',
    '    """Create an incident ticket. Returns ticket_id."""',
    '    # TODO: store in _tickets, return {"ticket_id": ..., "title": ..., "severity": ...}',
    '    pass',
    '',
    '',
    'if __name__ == "__main__":',
    '    mcp.run()',
]

with open('devops_mcp_server.py', 'w') as f:
    f.write('\n'.join(DEVOPS_LINES))
print('✅ devops_mcp_server.py scaffolded — fill in the TODOs!')


In [ ]:
# ── Unit-test the server tools directly ───────────────────────────────────
import importlib.util, sys as _sys

def test_devops_server() -> tuple:
    spec = importlib.util.spec_from_file_location("devops_mcp", "devops_mcp_server.py")
    if spec is None:
        return False, "File not found"
    mod = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(mod)
    except Exception as e:
        return False, f"Import error: {e}"

    required = ["get_server_status", "run_health_check", "list_alerts", "create_incident_ticket"]
    for fn_name in required:
        fn = getattr(mod, fn_name, None)
        if fn is None:
            return False, f"{fn_name} not found"
        if not fn.__doc__:
            return False, f"{fn_name} missing docstring"

    # Functional check
    try:
        ticket = mod.create_incident_ticket("DB down", "high", "Unresponsive for 5 min")
        if not ticket or "ticket_id" not in ticket:
            return False, "create_incident_ticket did not return ticket_id"
    except Exception as e:
        return False, f"create_incident_ticket raised: {e}"

    return True, ticket

ok, detail = test_devops_server()
print(f'Server test: {"✅" if ok else "❌"} — {detail}')

check([
    (os.path.exists("devops_mcp_server.py"), "devops_mcp_server.py exists"),
    (ok, f"All 4 tools pass unit test: {detail}"),
], exercise_id="04c-2")


In [ ]:
check([
    (os.path.exists("mcp_server_minimal.py"), "Minimal MCP server written"),
    (os.path.exists("devops_mcp_server.py"),  "DevOps MCP server exists"),
    (ok, "All tools pass unit tests"),
], exercise_id="04c-final")
progress()
